# Week 9: ARIA v6 Analysis

In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import json

# 載入 .env 檔案
load_dotenv()

# 取得 Sentinel-2 影像 ID
PRE_ITEM_ID = os.getenv('PRE_ITEM_ID')
MID_ITEM_ID = os.getenv('MID_ITEM_ID')
POST_ITEM_ID = os.getenv('POST_ITEM_ID')

# 定義分析用的雙重邊界框 (Bounding boxes)
# MATAIAN_BBOX (區域性)：提供完整研究區域的環境脈絡
MATAIAN_BBOX = {
    "west": float(os.getenv('BBOX_WEST', 121.28)),
    "south": float(os.getenv('BBOX_SOUTH', 23.56)),
    "east": float(os.getenv('BBOX_EAST', 121.52)),
    "north": float(os.getenv('BBOX_NORTH', 23.76))
}

# LAKE_BBOX_LONLAT (聚焦性)：專門用於湖泊範圍的精度評估
LAKE_BBOX_LONLAT = [121.27, 23.68, 121.32, 23.72]

# 驗證點資料路徑 (使用 raw string 處理 Windows 絕對路徑的斜線)
VALIDATION_POINTS_PATH = r"C:\Users\alvin\Desktop\Analysis and application of Remote sensing & geospatial information\Homework9\data\validation_points.geojson"

print(f"Pre ID: {PRE_ITEM_ID}")
print(f"Mid ID: {MID_ITEM_ID}")
print(f"Post ID: {POST_ITEM_ID}")
print("環境變數與邊界框設定完成！")

Pre ID: S2A_MSIL2A_請替換成你的災前影像ID
Mid ID: S2A_MSIL2A_請替換成你的災中影像ID
Post ID: S2A_MSIL2A_請替換成你的災後影像ID
環境變數與邊界框設定完成！


## 艦長日誌：步驟 1 - 載入 Sentinel-2 影像與 SCL 雲遮罩處理

在此步驟中，我們根據指定的 Sentinel-2 Item IDs (災前: 20250615, 災中: 20250911, 災後: 20251016) 提取衛星影像資料。
為了避免雲層與雲影在水體指數 (NDWI) 中造成「幽靈水 (phantom water)」的嚴重誤差，我們優先載入 SCL (Scene Classification Layer) 波段。我們將有效類別 `SCL_CLEAR_CLASSES = [2, 4, 5, 6, 7, 11]`（涵蓋植被、裸土、水體、雪、陰影、暗區）篩選出來，建立三個時期的二元遮罩 (`valid_pre`, `valid_mid`, `valid_post`)。
最後，將這三個遮罩取交集 (`valid_intersection`)，確保後續參與計算的像素在三個時期都是不受雲層干擾的乾淨觀測點。

In [6]:
import numpy as np
import xarray as xr
from pystac_client import Client
import planetary_computer as pc
import odc.stac

# SCL 有效類別定義：2=暗區, 4=植被, 5=裸土, 6=水體, 7=未分類/陰影, 11=雪/冰
SCL_CLEAR_CLASSES = [2, 4, 5, 6, 7, 11]

# 定義需要的波段 (SCL, Blue, Green, Red, NIR, SWIR)
# Sentinel-2 對應波段：SCL, B02, B03, B04, B08, B11
BANDS_OF_INTEREST = ["SCL", "B02", "B03", "B04", "B08", "B11"]

# 建立與 STAC Catalog 的連線 (此處以 Microsoft Planetary Computer 為例)
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace
)

def load_sentinel2_item(item_id, bbox_dict):
    """
    透過 STAC ID 與 Bounding Box 載入 Sentinel-2 L2A 影像波段。
    """
    # 搜尋特定的 Item ID
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        ids=[item_id]
    )
    items = list(search.items())
    if not items:
        raise ValueError(f"找不到影像 ID: {item_id}")
    
    # 定義經緯度邊界 [minx, miny, maxx, maxy]
    bbox = [bbox_dict['west'], bbox_dict['south'], bbox_dict['east'], bbox_dict['north']]
    
    # 使用 odc.stac 載入資料並裁切至 BBOX 範圍，設定解析度為 10m
    data = odc.stac.load(
        items,
        bands=BANDS_OF_INTEREST,
        bbox=bbox,
        resolution=10,
        crs="EPSG:32651", # 台灣東部所屬的 UTM 投影帶 (WGS84 / UTM zone 51N)
        chunks={"x": 2048, "y": 2048}
    )
    
    # 去除時間維度，回傳單一時間點的二維空間陣列
    return data.squeeze("time")

# ==========================================
# 1. 載入三個時期的影像資料
# ==========================================
print("正在載入災前影像 (Pre)...")
ds_pre = load_sentinel2_item(PRE_ITEM_ID, MATAIAN_BBOX)

print("正在載入災中影像 (Mid)...")
ds_mid = load_sentinel2_item(MID_ITEM_ID, MATAIAN_BBOX)

print("正在載入災後影像 (Post)...")
ds_post = load_sentinel2_item(POST_ITEM_ID, MATAIAN_BBOX)

# ==========================================
# 2. 實作 SCL 雲遮罩邏輯
# ==========================================
def stream_scl(scl_dataarray):
    """
    輸入 xarray.DataArray 格式的 SCL 波段，回傳二元遮罩 (True 代表有效)。
    """
    # 使用 xarray 的 isin 函數判斷像元值是否在有效清單中
    valid_mask = scl_dataarray.isin(SCL_CLEAR_CLASSES)
    return valid_mask

print("正在計算 SCL 雲遮罩...")
valid_pre = stream_scl(ds_pre.SCL)
valid_mid = stream_scl(ds_mid.SCL)
valid_post = stream_scl(ds_post.SCL)

# 建立交集遮罩 (三個時期皆為 True 才保留)
valid_intersection = valid_pre & valid_mid & valid_post

print(f"遮罩計算完成！有效像元數量: {int(valid_intersection.sum().compute())} pixels")
print("步驟 1 執行完畢，等待下一步指示。")

ModuleNotFoundError: No module named 'odc'